# Create training/validation/test splits (leave-one-sample-out)

This notebook reads `pork_shoulder_pairs_combined.csv` and creates two experiments:

- `exp1`: train = sample 1 (validation = 15% stratified by `label` if possible), test = sample 2
- `exp2`: train = sample 2 (validation = 15% stratified by `label` if possible), test = sample 1

Splits use only the `roi_file` (and an added `roi_path`) as the image reference.

In [ ]:
import math
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

# -------------------------------
# Configuration
# -------------------------------
LOCAL_CSV = Path('pork_shoulder_combined_roi.csv')
PUBLIC_CSV = Path('Public Dataset/_classes_combined_grouped_interval_sampled_20260430_193409.csv')
COMBINED_OUT = Path('_classes_combined_grouped.csv')
OUT_DIR = Path('cross_rotation_splits')

RANDOM_STATE = 42
VAL_SIZE = 0.15

OUTPUT_COLUMNS = [
    'file_name',
    'roi_file',
    'roi_path',
    'meat_part',
    'sample_number',
    'time',
    'label',
    'folder',
]


def normalize_label(value):
    text = str(value).strip().lower()
    norm = text.replace('_', ' ').replace('-', ' ')
    norm = ' '.join(norm.split())

    if norm == 'fresh':
        return 'fresh'
    if norm in {'not fresh', 'notfresh', 'half fresh', 'halffresh', 'semi fresh', 'semifresh'}:
        return 'not fresh'
    if norm in {'spoiled', 'spoil', 'spoilt', 'rotten'}:
        return 'spoiled'
    return norm


def strip_roi_suffix(name):
    name = str(name)
    suffix = '_roi'
    dot = name.rfind('.')
    if dot == -1:
        return name[:-len(suffix)] if name.endswith(suffix) else name
    stem = name[:dot]
    ext = name[dot:]
    if stem.endswith(suffix):
        stem = stem[:-len(suffix)]
    return f'{stem}{ext}'


def build_local_roi_path(row):
    sample = int(row['sample_number'])
    folder = str(row['folder']).strip()
    roi_file = str(row['roi_file']).strip()
    fallback_file = strip_roi_suffix(roi_file)
    sample_dir = f'Pork Shoulder - sample {sample}'

    candidates = [
        Path('processed_images') / sample_dir / folder / roi_file,
        Path('processed_images') / sample_dir / folder / fallback_file,
        Path('processed_image') / sample_dir / folder / roi_file,
        Path('processed_image') / sample_dir / folder / fallback_file,
        Path(sample_dir) / folder / roi_file,
        Path(sample_dir) / folder / fallback_file,
    ]

    for candidate in candidates:
        if candidate.exists():
            return str(candidate)

    return str(candidates[0])


def build_public_roi_path(row):
    filename = str(row['roi_file']).strip()
    label_folder = normalize_label(row['label'])

    candidates = [
        Path('Public Dataset') / label_folder / filename,
        Path('Public Dataset') / filename,
    ]

    for candidate in candidates:
        if candidate.exists():
            return str(candidate)

    return str(candidates[0])


def print_label_distribution(df, split_name):
    total = len(df)
    print(f'\n{split_name}: {total} rows')
    if total == 0:
        print('  - Empty split')
        return

    counts = df['label'].value_counts(dropna=False)
    for label, count in counts.items():
        pct = (count / total) * 100
        print(f'  - {label}: {count} ({pct:.2f}%)')

    samples = sorted(df['sample_number'].dropna().astype(int).unique().tolist())
    print(f'  sample_number values: {samples}')


def split_train_val(trainval_df, val_size=0.15, random_state=42):
    labels = trainval_df['label']
    n_rows = len(trainval_df)
    n_classes = labels.nunique()
    min_class = labels.value_counts().min()
    n_val = math.ceil(n_rows * val_size)

    can_stratify = min_class >= 2 and n_val >= n_classes

    if can_stratify:
        try:
            train_df, val_df = train_test_split(
                trainval_df,
                test_size=val_size,
                stratify=labels,
                random_state=random_state,
                shuffle=True,
            )
            return train_df.reset_index(drop=True), val_df.reset_index(drop=True)
        except ValueError as exc:
            print(f'WARNING: stratified split failed ({exc}); falling back to random split.')

    else:
        print(
            'WARNING: stratified split not possible '
            f'(rows={n_rows}, classes={n_classes}, min_class={min_class}, val_rows={n_val}); '
            'falling back to random split.'
        )

    train_df, val_df = train_test_split(
        trainval_df,
        test_size=val_size,
        random_state=random_state,
        shuffle=True,
    )
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True)


def run_experiment(df_all, exp_name, train_samples, test_sample):
    trainval_df = df_all[df_all['sample_number'].isin(train_samples)].copy()
    test_df = df_all[df_all['sample_number'] == test_sample].copy()

    if test_df.empty:
        print(f'WARNING: {exp_name} test split is empty for sample {test_sample}.')
    if trainval_df.empty:
        raise ValueError(f'{exp_name} train/val pool is empty for samples {train_samples}.')

    train_df, val_df = split_train_val(
        trainval_df,
        val_size=VAL_SIZE,
        random_state=RANDOM_STATE,
    )

    # Leakage checks
    train_samples_set = set(train_df['sample_number'].astype(int).unique())
    val_samples_set = set(val_df['sample_number'].astype(int).unique())
    test_samples_set = set(test_df['sample_number'].astype(int).unique())

    if test_sample in train_samples_set or test_sample in val_samples_set:
        raise AssertionError(f'Leakage detected in {exp_name}: test sample appears in train/val.')

    if not train_samples_set.issubset(set(train_samples)) or not val_samples_set.issubset(set(train_samples)):
        raise AssertionError(f'Unexpected sample numbers found in train/val for {exp_name}.')

    print(f'\n===== {exp_name.upper()} =====')
    print(f'Train samples: {train_samples} | Test sample: {test_sample}')
    print_label_distribution(train_df, f'{exp_name}_train')
    print_label_distribution(val_df, f'{exp_name}_val')
    print_label_distribution(test_df, f'{exp_name}_test')

    train_out = OUT_DIR / f'{exp_name}_train.csv'
    val_out = OUT_DIR / f'{exp_name}_val.csv'
    test_out = OUT_DIR / f'{exp_name}_test.csv'

    train_df[OUTPUT_COLUMNS].to_csv(train_out, index=False)
    val_df[OUTPUT_COLUMNS].to_csv(val_out, index=False)
    test_df[OUTPUT_COLUMNS].to_csv(test_out, index=False)

    print(f'Saved: {train_out}')
    print(f'Saved: {val_out}')
    print(f'Saved: {test_out}')


# -------------------------------
# Load + standardize local dataset
# -------------------------------
local_df = pd.read_csv(LOCAL_CSV)
print(f'Loaded local dataset: {LOCAL_CSV} ({len(local_df)} rows)')

required_local_base = {'file_name', 'meat_part', 'sample_number', 'time', 'label', 'folder'}
missing_local_base = sorted(required_local_base - set(local_df.columns))
if missing_local_base:
    raise ValueError(f'Local CSV missing required columns: {missing_local_base}')

if 'roi_file' not in local_df.columns:
    local_df['roi_file'] = local_df['file_name']

# If local file_name is ROI filename, keep compatibility by deriving original-like file_name.
local_df['file_name'] = local_df['file_name'].astype(str).map(strip_roi_suffix)
local_df['roi_file'] = local_df['roi_file'].astype(str)
local_df['label'] = local_df['label'].map(normalize_label)
local_df['sample_number'] = local_df['sample_number'].astype(int)
local_df['roi_path'] = local_df.apply(build_local_roi_path, axis=1)

local_df = local_df[[
    'file_name', 'roi_file', 'roi_path', 'meat_part',
    'sample_number', 'time', 'label', 'folder'
]].copy()


# -------------------------------
# Load + standardize public dataset
# -------------------------------
public_df = pd.read_csv(PUBLIC_CSV)
print(f'Loaded public dataset: {PUBLIC_CSV} ({len(public_df)} rows)')

required_public = {'filename', 'label'}
missing_public = sorted(required_public - set(public_df.columns))
if missing_public:
    raise ValueError(f'Public CSV missing required columns: {missing_public}')

public_df = public_df.rename(columns={'filename': 'file_name'}).copy()
public_df['roi_file'] = public_df['file_name']
public_df['meat_part'] = 'Different Pork Meat'
public_df['sample_number'] = 3
public_df['time'] = 'unknown'
public_df['label'] = public_df['label'].map(normalize_label)
public_df['folder'] = 'public_dataset'
public_df['roi_path'] = public_df.apply(build_public_roi_path, axis=1)

public_df = public_df[[
    'file_name', 'roi_file', 'roi_path', 'meat_part',
    'sample_number', 'time', 'label', 'folder'
]].copy()


# -------------------------------
# Merge + checks
# -------------------------------
combined_df = pd.concat([local_df, public_df], ignore_index=True)
combined_df = combined_df[OUTPUT_COLUMNS].copy()

print(f'\nCombined dataset rows: {len(combined_df)}')
print('Combined sample_number values:', sorted(combined_df['sample_number'].astype(int).unique().tolist()))
print_label_distribution(combined_df, 'combined_dataset')

missing_roi_mask = ~combined_df['roi_path'].map(lambda p: Path(p).exists())
missing_roi_df = combined_df[missing_roi_mask]
print(f'\nMissing roi_path files: {len(missing_roi_df)}')
if not missing_roi_df.empty:
    print(missing_roi_df[['sample_number', 'label', 'roi_path']].head(50).to_string(index=False))

combined_df.to_csv(COMBINED_OUT, index=False)
print(f'\nSaved merged dataset: {COMBINED_OUT}')


# -------------------------------
# Cross-rotation experiments
# -------------------------------
OUT_DIR.mkdir(parents=True, exist_ok=True)

experiments = [
    ('exp1', [1, 2], 3),
    ('exp2', [2, 3], 1),
    ('exp3', [3, 1], 2),
]

for exp_name, train_samples, test_sample in experiments:
    run_experiment(combined_df, exp_name, train_samples, test_sample)

print('\nAll splits generated successfully.')


